In [1]:
from __future__ import annotations

import math
import os
import re
import warnings
from contextlib import closing
from dataclasses import dataclass
from datetime import date, timedelta
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None

try:
    display
except NameError:
    def display(obj):
        print(obj)

warnings.filterwarnings("ignore", category=RuntimeWarning)
try:
    warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)
except Exception:
    pass

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)


In [2]:
# 安装依赖
!pip install -U tqsdk pandas numpy statsmodels arch matplotlib openpyxl

In [3]:
PROJECT_DIR = Path("main_continuous_daily_trend_momentum_reversal_research")
RAW_MAIN_DIR = PROJECT_DIR / "raw" / "tq_main_continuous_daily"
PROCESSED_DIR = PROJECT_DIR / "processed"
RESULT_DIR = PROJECT_DIR / "results"
FIG_DIR = PROJECT_DIR / "figures"

for p in [RAW_MAIN_DIR, PROCESSED_DIR, RESULT_DIR, FIG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# 运行开关
RUN_TQ_DOWNLOAD = True
AUTO_DOWNLOAD_IF_MISSING = True
OVERWRITE_RAW_CSV = True
RUN_PLOTS = False

# 日线参数
DAILY_KLINE_DATA_LENGTH = 8000
DAILY_KLINE_DUR_SEC = 24 * 60 * 60

# 研究区间
START_DATE = date(1990, 1, 1)
END_DATE = date.today()
LOCAL_TIMEZONE = "Asia/Shanghai"

# 品种起始过滤日期
PRODUCT_START_DATE = {
    "M": date(2000, 1, 1),
    "SP": date(2018, 11, 27),
    "FG": date(2012, 12, 3),
    "SA": date(2019, 12, 6),

    "AG": date(2012, 5, 10),
    "V": date(2009, 5, 25),
    "LH": date(2021, 1, 8),
    "EB": date(2019, 9, 26),
    "MA": date(2011, 10, 28),
    "CF": date(2004, 6, 1),
    "SR": date(2006, 1, 6),
}

MIN_OHLCV_VOLUME = 0
MIN_SIGMA = 1e-6
VOL_BASE_LOOKBACK = 20
EWMA_LAMBDA = 0.94

H_RET_LIST = [1, 2, 3, 5, 10, 20, 40, 60]
H_TREND_LIST = [10, 20, 40, 60]
J_LIST = [2, 3, 5, 10, 20, 40, 60]
K_LIST = [1, 2, 3, 5, 10, 20]
L_LIST = [0, 2, 3, 5, 10, 20]

TREND_T_THRESHOLD = 2.0
TREND_FIT_THRESHOLD = 0.20
TREND_OVEREXT_Q = 0.90
ATR_STOP_LOOKBACK = 20
ATR_STOP_MULT = 2.5

MOM_Z_THRESHOLD = 1.0
MCS_THRESHOLD = 0.58
SHOCK_Z_THRESHOLD = 2.0
VOLUME_CONFIRM_Z = 0.25

FAST_REV_Z = 2.0
REV_SIGNAL_MIN_ABS = 1e-12

MIN_MODEL_N = 60
MIN_CONFIRMED_N = 120
MIN_T_CAND = 1.0
MIN_T_CONF = 2.0
MIN_HIT_RATE = 0.50
BOOTSTRAP_BLOCK = 5
BOOTSTRAP_N = 400
RANDOM_SEED = 20260630

rng = np.random.default_rng(RANDOM_SEED)


In [4]:
@dataclass(frozen=True)
class ProductSpec:
    commodity: str
    name: str
    main_symbol: str


PRODUCTS = {
    "M": ProductSpec("M", "豆粕", "KQ.m@DCE.m"),
    "SP": ProductSpec("SP", "纸浆", "KQ.m@SHFE.sp"),
    "FG": ProductSpec("FG", "玻璃", "KQ.m@CZCE.FG"),
    "SA": ProductSpec("SA", "纯碱", "KQ.m@CZCE.SA"),

    "AG": ProductSpec("AG", "沪银", "KQ.m@SHFE.ag"),
    "V": ProductSpec("V", "PVC", "KQ.m@DCE.v"),
    "LH": ProductSpec("LH", "生猪", "KQ.m@DCE.lh"),
    "EB": ProductSpec("EB", "苯乙烯", "KQ.m@DCE.eb"),
    "MA": ProductSpec("MA", "甲醇", "KQ.m@CZCE.MA"),
    "CF": ProductSpec("CF", "棉花", "KQ.m@CZCE.CF"),
    "SR": ProductSpec("SR", "白糖", "KQ.m@CZCE.SR"),
}

# 天勤账号
TQ_USERNAME = os.getenv("TQ_USERNAME", "")
TQ_PASSWORD = os.getenv("TQ_PASSWORD", "")

print("研究品种：")
for k, v in PRODUCTS.items():
    print(f"  {k}: {v.main_symbol}, name={v.name}")

print("\n是否在 Cell 04 主动下载：", RUN_TQ_DOWNLOAD)
print("缺失数据时是否自动下载：", AUTO_DOWNLOAD_IF_MISSING)
print("当前工作目录：", Path.cwd().resolve())
print("主力连续合约 CSV 目录：", RAW_MAIN_DIR.resolve())


研究品种：
  M: KQ.m@DCE.m, name=豆粕
  SP: KQ.m@SHFE.sp, name=纸浆
  FG: KQ.m@CZCE.FG, name=玻璃
  SA: KQ.m@CZCE.SA, name=纯碱
  AG: KQ.m@SHFE.ag, name=沪银
  V: KQ.m@DCE.v, name=PVC
  LH: KQ.m@DCE.lh, name=生猪
  EB: KQ.m@DCE.eb, name=苯乙烯
  MA: KQ.m@CZCE.MA, name=甲醇
  CF: KQ.m@CZCE.CF, name=棉花
  SR: KQ.m@CZCE.SR, name=白糖

是否在 Cell 04 主动下载： True
缺失数据时是否自动下载： True
当前工作目录： /home/zilinm2
主力连续合约 CSV 目录： /home/zilinm2/main_continuous_daily_trend_momentum_reversal_research/raw/tq_main_continuous_daily


In [5]:
def safe_filename(symbol: str) -> str:
    return re.sub(r"[^A-Za-z0-9_]+", "_", symbol)


def main_csv_path(commodity: str, symbol: str) -> Path:
    return RAW_MAIN_DIR / f"{commodity}_{safe_filename(symbol)}_daily.csv"


def has_tq_credentials() -> bool:
    return bool(str(TQ_USERNAME).strip()) and bool(str(TQ_PASSWORD).strip())


def create_tq_api():
    from tqsdk import TqApi, TqAuth

    if not has_tq_credentials():
        raise RuntimeError(
            "缺少天勤账号。请设置环境变量 TQ_USERNAME / TQ_PASSWORD，"
            "或在 Cell 03 中填写 TQ_USERNAME / TQ_PASSWORD。"
        )

    return TqApi(auth=TqAuth(TQ_USERNAME, TQ_PASSWORD))


def parse_numeric_datetime_series(x: pd.Series) -> pd.Series:
    """
    解析 TqSdk K 线 datetime。
    关键点：get_kline_serial 前部可能有 datetime=0 的占位行，不能参与时间单位判断。
    """
    x_num = pd.to_numeric(x, errors="coerce")
    positive = x_num[x_num > 0].dropna()

    out = pd.Series(pd.NaT, index=x.index, dtype="datetime64[ns]")

    if positive.empty:
        return out

    med = positive.median()
    valid = x_num > 0

    if 19000101 <= med <= 21001231:
        as_text = x_num.loc[valid].round().astype("Int64").astype(str).str.zfill(8)
        out.loc[valid] = pd.to_datetime(as_text, format="%Y%m%d", errors="coerce")
        return out

    if med > 1e17:
        unit = "ns"
    elif med > 1e14:
        unit = "us"
    elif med > 1e11:
        unit = "ms"
    else:
        unit = "s"

    dt = pd.to_datetime(x_num.loc[valid], unit=unit, errors="coerce", utc=True)
    out.loc[valid] = dt.dt.tz_convert(LOCAL_TIMEZONE).dt.tz_localize(None)

    return out


def parse_datetime_series(s: pd.Series) -> pd.Series:
    if pd.api.types.is_numeric_dtype(s):
        return parse_numeric_datetime_series(s)

    text = s.astype(str).str.strip()
    numeric = pd.to_numeric(text, errors="coerce")

    if numeric.notna().mean() >= 0.80:
        return parse_numeric_datetime_series(numeric)

    return pd.to_datetime(text, errors="coerce")


def normalize_kline_serial_to_daily(
    kline_df: pd.DataFrame,
    commodity: str,
    symbol: str,
    csv_path: Path,
) -> pd.DataFrame:
    """
    将 TqSdk 主力连续合约日线 K 线标准化为研究用 OHLCV。
    """
    if kline_df is None or len(kline_df) == 0:
        return pd.DataFrame()

    raw = kline_df.copy()
    raw.columns = [str(c).strip() for c in raw.columns]
    lower_map = {c.lower(): c for c in raw.columns}

    required = ["datetime", "open", "high", "low", "close", "volume"]
    missing = [c for c in required if c not in lower_map]
    if missing:
        raise ValueError(f"日线 K 线缺少字段：{missing}；当前字段：{list(raw.columns)}")

    df = pd.DataFrame()
    df["datetime"] = parse_datetime_series(raw[lower_map["datetime"]])
    df["date"] = df["datetime"].dt.normalize()

    for c in ["open", "high", "low", "close", "volume"]:
        df[c] = pd.to_numeric(raw[lower_map[c]], errors="coerce")

    df["commodity"] = commodity
    df["commodity_name"] = PRODUCTS[commodity].name
    df["main_symbol"] = symbol
    df["data_source"] = "TQ_MAIN_CONTINUOUS_KLINE_DAILY"
    df["csv_path"] = str(csv_path)

    start_ts = pd.Timestamp(max(PRODUCT_START_DATE.get(commodity, START_DATE), START_DATE))
    end_ts = pd.Timestamp(END_DATE)

    valid = (
        df["date"].notna()
        & (df["date"] >= start_ts)
        & (df["date"] <= end_ts)
        & (df[["open", "high", "low", "close"]] > 0).all(axis=1)
        & (df["volume"].fillna(-1) >= MIN_OHLCV_VOLUME)
        & (df["high"] >= df[["open", "close", "low"]].max(axis=1))
        & (df["low"] <= df[["open", "close", "high"]].min(axis=1))
    )

    df = df.loc[valid].copy()

    if df.empty:
        return pd.DataFrame()

    df = df.sort_values(["date", "datetime"]).drop_duplicates(["date"], keep="last")

    keep_cols = [
        "date", "datetime", "commodity", "commodity_name", "main_symbol", "data_source",
        "open", "high", "low", "close", "volume", "csv_path",
    ]

    return df[keep_cols].reset_index(drop=True)


def tq_wait_update_safe(api, deadline: int = 1) -> None:
    try:
        api.wait_update(deadline=deadline)
    except TypeError:
        api.wait_update()


def fetch_one_daily_kline_csv(
    api,
    commodity: str,
    symbol: str,
    csv_path: Path,
    overwrite: bool = False,
    data_length: int = DAILY_KLINE_DATA_LENGTH,
) -> Tuple[bool, str, int]:
    """
    直接抓取 TqSdk 主力连续合约日线 K 线并保存 CSV。
    """
    if csv_path.exists() and not overwrite:
        existing = pd.read_csv(csv_path)
        return True, "existing", int(len(existing))

    csv_path.parent.mkdir(parents=True, exist_ok=True)

    try:
        klines = api.get_kline_serial(
            symbol,
            DAILY_KLINE_DUR_SEC,
            data_length=data_length,
        )

        if hasattr(api, "is_serial_ready"):
            wait_count = 0
            while not api.is_serial_ready(klines) and wait_count < 200:
                tq_wait_update_safe(api, deadline=1)
                wait_count += 1
        else:
            for _ in range(20):
                tq_wait_update_safe(api, deadline=1)

        daily_panel = normalize_kline_serial_to_daily(
            klines.copy(),
            commodity,
            symbol,
            csv_path,
        )

        if daily_panel.empty:
            return False, "日线 K 线为空或清洗后无有效 OHLCV", 0

        daily_panel.to_csv(csv_path, index=False)
        return True, "", int(len(daily_panel))

    except Exception as e:
        return False, str(e), 0


def download_main_continuous_for_products(
    products: Dict[str, ProductSpec] = PRODUCTS,
    overwrite: bool = OVERWRITE_RAW_CSV,
) -> pd.DataFrame:
    rows = []

    with closing(create_tq_api()) as api:
        for commodity, spec in products.items():
            csv_path = main_csv_path(commodity, spec.main_symbol)

            ok, err, rows_count = fetch_one_daily_kline_csv(
                api=api,
                commodity=commodity,
                symbol=spec.main_symbol,
                csv_path=csv_path,
                overwrite=overwrite,
            )

            rows.append({
                "commodity": commodity,
                "symbol": spec.main_symbol,
                "csv_path": str(csv_path),
                "ok": ok,
                "rows": rows_count,
                "error": err,
            })

            print(f"[主连日线] {commodity}: ok={ok}, rows={rows_count}, csv={csv_path}")

    report = pd.DataFrame(rows)
    report.to_csv(RESULT_DIR / "tq_main_continuous_daily_download_report.csv", index=False)
    return report


if RUN_TQ_DOWNLOAD:
    if has_tq_credentials():
        main_download_report = download_main_continuous_for_products()
        display(main_download_report)
    else:
        print("RUN_TQ_DOWNLOAD=True，但尚未填写天勤账号。请在 Cell 03 填写账号后重跑 Cell 03-04。")
else:
    print("RUN_TQ_DOWNLOAD=False。若本地 CSV 缺失且已填写天勤账号，后续 Cell 会自动抓取日线 K 线。")

在使用天勤量化之前，默认您已经知晓并同意以下免责条款，如果不同意请立即停止使用：https://www.shinnytech.com/blog/disclaimer/


2026-06-30 07:04:54 -     INFO - 通知 : 与 wss://free-api.shinnytech.com/t/nfmd/front/mobile 的网络连接已建立
[主连日线] M: ok=True, rows=2544, csv=main_continuous_daily_trend_momentum_reversal_research/raw/tq_main_continuous_daily/M_KQ_m_DCE_m_daily.csv
[主连日线] SP: ok=True, rows=1838, csv=main_continuous_daily_trend_momentum_reversal_research/raw/tq_main_continuous_daily/SP_KQ_m_SHFE_sp_daily.csv
[主连日线] FG: ok=True, rows=2544, csv=main_continuous_daily_trend_momentum_reversal_research/raw/tq_main_continuous_daily/FG_KQ_m_CZCE_FG_daily.csv
[主连日线] SA: ok=True, rows=1588, csv=main_continuous_daily_trend_momentum_reversal_research/raw/tq_main_continuous_daily/SA_KQ_m_CZCE_SA_daily.csv
[主连日线] AG: ok=True, rows=2544, csv=main_continuous_daily_trend_momentum_reversal_research/raw/tq_main_continuous_daily/AG_KQ_m_SHFE_ag_daily.csv
[主连日线] V: ok=True, rows=2544, csv=main_continuous_daily_trend_momentum_reversal_research/raw/tq_main_continuous_daily/V_KQ_m_DCE_v_daily.csv
[主连日线] LH: ok=True, rows=1323, csv=main

,commodity,symbol,csv_path,ok,rows,error
0,M,KQ.m@DCE.m,main_continuous_daily_trend_momentum_reversal_...,True,2544,
1,SP,KQ.m@SHFE.sp,main_continuous_daily_trend_momentum_reversal_...,True,1838,
2,FG,KQ.m@CZCE.FG,main_continuous_daily_trend_momentum_reversal_...,True,2544,
3,SA,KQ.m@CZCE.SA,main_continuous_daily_trend_momentum_reversal_...,True,1588,
4,AG,KQ.m@SHFE.ag,main_continuous_daily_trend_momentum_reversal_...,True,2544,
5,V,KQ.m@DCE.v,main_continuous_daily_trend_momentum_reversal_...,True,2544,
6,LH,KQ.m@DCE.lh,main_continuous_daily_trend_momentum_reversal_...,True,1323,
7,EB,KQ.m@DCE.eb,main_continuous_daily_trend_momentum_reversal_...,True,1634,
8,MA,KQ.m@CZCE.MA,main_continuous_daily_trend_momentum_reversal_...,True,2544,
9,CF,KQ.m@CZCE.CF,main_continuous_daily_trend_momentum_reversal_...,True,2544,


In [6]:
def normalize_tq_daily_csv(csv_path: Path, commodity: str, symbol: str) -> pd.DataFrame:
    raw = pd.read_csv(csv_path)
    raw.columns = [str(c).strip() for c in raw.columns]
    lower_map = {c.lower(): c for c in raw.columns}

    dt_col = None
    for cand in ["datetime", "date", "time", "trading_day"]:
        if cand in lower_map:
            dt_col = lower_map[cand]
            break
    if dt_col is None:
        raise ValueError(f"{csv_path} 没有 datetime/date/time/trading_day 字段")

    col_map = {}
    for std in ["open", "high", "low", "close", "volume"]:
        if std in lower_map:
            col_map[lower_map[std]] = std

    missing = sorted(set(["open", "high", "low", "close", "volume"]) - set(col_map.values()))
    if missing:
        raise ValueError(f"{csv_path} 缺少 OHLCV 字段：{missing}")

    df = raw[[dt_col] + list(col_map.keys())].rename(columns={dt_col: "datetime", **col_map})
    df["datetime"] = parse_datetime_series(df["datetime"])
    df["date"] = df["datetime"].dt.normalize()

    for c in ["open", "high", "low", "close", "volume"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df["commodity"] = commodity
    df["commodity_name"] = PRODUCTS[commodity].name
    df["main_symbol"] = symbol
    if "data_source" in lower_map:
        source_val = str(raw[lower_map["data_source"]].dropna().iloc[0]) if raw[lower_map["data_source"]].notna().any() else "TQ_MAIN_CONTINUOUS_KLINE_DAILY"
    else:
        source_val = "TQ_MAIN_CONTINUOUS_KLINE_DAILY"
    df["data_source"] = source_val
    df["csv_path"] = str(csv_path)

    keep_cols = [
        "date", "datetime", "commodity", "commodity_name", "main_symbol",
        "data_source", "open", "high", "low", "close", "volume", "csv_path",
    ]
    df = df[keep_cols].dropna(subset=["date", "open", "high", "low", "close"])
    df = df.sort_values(["commodity", "date"]).drop_duplicates(["commodity", "date"], keep="last")
    return df.reset_index(drop=True)


def list_main_csvs(commodity: str) -> List[Path]:
    spec = PRODUCTS[commodity]
    expected = main_csv_path(commodity, spec.main_symbol)
    if expected.exists():
        return [expected]
    return sorted(
        RAW_MAIN_DIR.glob(f"{commodity}_*_daily.csv"),
        key=lambda p: (p.stat().st_mtime, p.name),
    )


In [7]:
def missing_required_commodities(df: pd.DataFrame) -> List[str]:
    present = set(df["commodity"].dropna().unique()) if "commodity" in df.columns and not df.empty else set()
    return [commodity for commodity in PRODUCTS if commodity not in present]


def load_main_continuous_panel() -> pd.DataFrame:
    frames = []
    errors = []

    for commodity, spec in PRODUCTS.items():
        paths = list_main_csvs(commodity)
        if not paths:
            print(f"[WARN] 未发现 {commodity} 的主力连续合约日线 CSV")
            continue
        path = paths[-1]
        try:
            df = normalize_tq_daily_csv(path, commodity, symbol=spec.main_symbol)
            if len(df) > 0:
                frames.append(df)
        except Exception as e:
            errors.append({"commodity": commodity, "csv_path": str(path), "error": str(e)})

    if errors:
        pd.DataFrame(errors).to_csv(RESULT_DIR / "load_main_continuous_errors.csv", index=False)
        print(f"[WARN] 主力连续合约日线 CSV 读取错误数量：{len(errors)}")

    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


def load_research_panel() -> pd.DataFrame:
    main_panel = load_main_continuous_panel()
    missing = missing_required_commodities(main_panel)

    if missing and AUTO_DOWNLOAD_IF_MISSING:
        if has_tq_credentials():
            print(f"[AUTO] 主力连续合约日线 CSV 不完整，开始补齐：{missing}")
            report = download_main_continuous_for_products({c: PRODUCTS[c] for c in missing})
            display(report)
            main_panel = load_main_continuous_panel()
            missing = missing_required_commodities(main_panel)
        else:
            print(
                f"[AUTO] 主力连续合约日线 CSV 不完整，缺失 {missing}，且未填写天勤账号。"
                "请在 Cell 03 填写 TQ_USERNAME / TQ_PASSWORD 后，重新运行 Cell 03-06。"
            )

    if not missing:
        print(f"[OK] 已载入主力连续合约数据：{len(main_panel):,} 行，{main_panel['commodity'].nunique()} 个标的")
        return main_panel

    raise RuntimeError(
        f"未能载入完整 4 个主力连续合约日线数据，缺失：{missing}。请处理以下任一项：\n"
        "1. 在 Cell 03 填写 TQ_USERNAME / TQ_PASSWORD，并重新运行 Cell 03-06；或\n"
        "2. 设置 RUN_TQ_DOWNLOAD=True，并从 Cell 02 开始重新运行；或\n"
        f"3. 将 4 个日线 CSV 放入 {RAW_MAIN_DIR}/。"
    )


panel_raw = load_research_panel()
display(panel_raw.head())
display(panel_raw.groupby(["commodity", "main_symbol"]).agg(
    rows=("date", "size"),
    start=("date", "min"),
    end=("date", "max"),
))


[OK] 已载入主力连续合约数据：24,191 行，11 个标的


,date,datetime,commodity,commodity_name,main_symbol,data_source,open,high,low,close,volume,csv_path
0,2016-01-05,2016-01-05,M,豆粕,KQ.m@DCE.m,TQ_MAIN_CONTINUOUS_KLINE_DAILY,2327.0,2347.0,2325.0,2343.0,569527.0,main_continuous_daily_trend_momentum_reversal_...
1,2016-01-06,2016-01-06,M,豆粕,KQ.m@DCE.m,TQ_MAIN_CONTINUOUS_KLINE_DAILY,2345.0,2352.0,2327.0,2347.0,683608.0,main_continuous_daily_trend_momentum_reversal_...
2,2016-01-07,2016-01-07,M,豆粕,KQ.m@DCE.m,TQ_MAIN_CONTINUOUS_KLINE_DAILY,2341.0,2386.0,2339.0,2376.0,987863.0,main_continuous_daily_trend_momentum_reversal_...
3,2016-01-08,2016-01-08,M,豆粕,KQ.m@DCE.m,TQ_MAIN_CONTINUOUS_KLINE_DAILY,2375.0,2407.0,2363.0,2404.0,989187.0,main_continuous_daily_trend_momentum_reversal_...
4,2016-01-11,2016-01-11,M,豆粕,KQ.m@DCE.m,TQ_MAIN_CONTINUOUS_KLINE_DAILY,2407.0,2413.0,2377.0,2383.0,745035.0,main_continuous_daily_trend_momentum_reversal_...


,,rows,start,end
commodity,main_symbol,,,
AG,KQ.m@SHFE.ag,2544,2016-01-05,2026-06-29
CF,KQ.m@CZCE.CF,2544,2016-01-05,2026-06-29
EB,KQ.m@DCE.eb,1634,2019-09-26,2026-06-29
FG,KQ.m@CZCE.FG,2544,2016-01-05,2026-06-29
LH,KQ.m@DCE.lh,1323,2021-01-08,2026-06-29
M,KQ.m@DCE.m,2544,2016-01-05,2026-06-29
MA,KQ.m@CZCE.MA,2544,2016-01-05,2026-06-29
SA,KQ.m@CZCE.SA,1588,2019-12-06,2026-06-29
SP,KQ.m@SHFE.sp,1838,2018-11-27,2026-06-29


In [8]:
def clean_ohlcv_panel(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out = out.sort_values(["commodity", "date"]).reset_index(drop=True)

    for c in ["open", "high", "low", "close", "volume"]:
        out[c] = pd.to_numeric(out[c], errors="coerce")

    bad = (
        out[["open", "high", "low", "close"]].isna().any(axis=1)
        | (out[["open", "high", "low", "close"]] <= 0).any(axis=1)
        | (out["volume"].fillna(-1) < MIN_OHLCV_VOLUME)
        | (out["high"] < out[["open", "close", "low"]].max(axis=1))
        | (out["low"] > out[["open", "close", "high"]].min(axis=1))
    )

    out["ohlcv_bad"] = bad.astype(float)
    out = out.loc[~bad].copy()

    out["series_age"] = out.groupby("commodity").cumcount() + 1
    out["series_len"] = out.groupby("commodity")["date"].transform("size")
    out["series_remain"] = out["series_len"] - out["series_age"]
    out["series_start"] = out.groupby("commodity")["date"].transform("min")
    out["series_end"] = out.groupby("commodity")["date"].transform("max")
    out["data_reliability"] = "MAIN_CONTINUOUS_DIAGNOSTIC"

    return out.sort_values(["commodity", "date"]).reset_index(drop=True)


panel = clean_ohlcv_panel(panel_raw)
missing_after_cleaning = missing_required_commodities(panel)
if missing_after_cleaning:
    raise RuntimeError(f"OHLCV 清洗后缺失品种：{missing_after_cleaning}。请检查对应主力连续合约日线 CSV。")

display(panel.head())
display(panel.groupby(["commodity", "data_reliability"]).agg(
    rows=("date", "size"),
    start=("date", "min"),
    end=("date", "max"),
))


,date,datetime,commodity,commodity_name,main_symbol,data_source,open,high,low,close,volume,csv_path,ohlcv_bad,series_age,series_len,series_remain,series_start,series_end,data_reliability
0,2016-01-05,2016-01-05,AG,沪银,KQ.m@SHFE.ag,TQ_MAIN_CONTINUOUS_KLINE_DAILY,3306.0,3330.0,3294.0,3313.0,287111.0,main_continuous_daily_trend_momentum_reversal_...,0.0,1,2544,2543,2016-01-05,2026-06-29,MAIN_CONTINUOUS_DIAGNOSTIC
1,2016-01-06,2016-01-06,AG,沪银,KQ.m@SHFE.ag,TQ_MAIN_CONTINUOUS_KLINE_DAILY,3314.0,3321.0,3296.0,3314.0,204526.0,main_continuous_daily_trend_momentum_reversal_...,0.0,2,2544,2542,2016-01-05,2026-06-29,MAIN_CONTINUOUS_DIAGNOSTIC
2,2016-01-07,2016-01-07,AG,沪银,KQ.m@SHFE.ag,TQ_MAIN_CONTINUOUS_KLINE_DAILY,3322.0,3385.0,3308.0,3357.0,488186.0,main_continuous_daily_trend_momentum_reversal_...,0.0,3,2544,2541,2016-01-05,2026-06-29,MAIN_CONTINUOUS_DIAGNOSTIC
3,2016-01-08,2016-01-08,AG,沪银,KQ.m@SHFE.ag,TQ_MAIN_CONTINUOUS_KLINE_DAILY,3367.0,3390.0,3345.0,3356.0,379737.0,main_continuous_daily_trend_momentum_reversal_...,0.0,4,2544,2540,2016-01-05,2026-06-29,MAIN_CONTINUOUS_DIAGNOSTIC
4,2016-01-11,2016-01-11,AG,沪银,KQ.m@SHFE.ag,TQ_MAIN_CONTINUOUS_KLINE_DAILY,3345.0,3362.0,3322.0,3349.0,283036.0,main_continuous_daily_trend_momentum_reversal_...,0.0,5,2544,2539,2016-01-05,2026-06-29,MAIN_CONTINUOUS_DIAGNOSTIC


,,rows,start,end
commodity,data_reliability,,,
AG,MAIN_CONTINUOUS_DIAGNOSTIC,2544,2016-01-05,2026-06-29
CF,MAIN_CONTINUOUS_DIAGNOSTIC,2544,2016-01-05,2026-06-29
EB,MAIN_CONTINUOUS_DIAGNOSTIC,1634,2019-09-26,2026-06-29
FG,MAIN_CONTINUOUS_DIAGNOSTIC,2544,2016-01-05,2026-06-29
LH,MAIN_CONTINUOUS_DIAGNOSTIC,1323,2021-01-08,2026-06-29
M,MAIN_CONTINUOUS_DIAGNOSTIC,2544,2016-01-05,2026-06-29
MA,MAIN_CONTINUOUS_DIAGNOSTIC,2544,2016-01-05,2026-06-29
SA,MAIN_CONTINUOUS_DIAGNOSTIC,1588,2019-12-06,2026-06-29
SP,MAIN_CONTINUOUS_DIAGNOSTIC,1838,2018-11-27,2026-06-29


In [9]:
def nw_tstat(x: Sequence[float], lags: Optional[int] = None) -> Tuple[float, float]:
    arr = pd.Series(x).dropna().astype(float).to_numpy()
    n = len(arr)
    if n < 3:
        return np.nan, np.nan
    mean = float(np.mean(arr))
    demeaned = arr - mean
    if lags is None:
        lags = int(np.floor(4 * (n / 100) ** (2 / 9)))
    lags = max(0, min(lags, n - 1))
    gamma0 = float(np.dot(demeaned, demeaned) / n)
    lrv = gamma0
    for lag in range(1, lags + 1):
        cov = float(np.dot(demeaned[lag:], demeaned[:-lag]) / n)
        weight = 1.0 - lag / (lags + 1.0)
        lrv += 2.0 * weight * cov
    se = math.sqrt(max(lrv, 0) / n) if n > 0 else np.nan
    t = mean / se if se and se > 0 else np.nan
    return mean, t


def block_bootstrap_p_positive(x: Sequence[float], block: int = BOOTSTRAP_BLOCK, n_boot: int = BOOTSTRAP_N) -> float:
    arr = pd.Series(x).dropna().astype(float).to_numpy()
    n = len(arr)
    if n < max(10, block):
        return np.nan
    blocks = [arr[i:i + block] for i in range(0, n - block + 1)]
    if not blocks:
        return np.nan
    means = []
    need_blocks = int(math.ceil(n / block))
    for _ in range(n_boot):
        sample = np.concatenate([blocks[i] for i in rng.integers(0, len(blocks), size=need_blocks)])[:n]
        means.append(np.mean(sample))
    return float(np.mean(np.asarray(means) <= 0))


def summarize_directed_returns(
    directed_y: Sequence[float],
    raw_future_ret: Optional[Sequence[float]] = None,
    k: Optional[int] = None,
    label: str = "",
) -> Dict[str, float]:
    y_raw = pd.to_numeric(
        pd.Series(directed_y).replace([np.inf, -np.inf], np.nan),
        errors="coerce",
    )

    if raw_future_ret is not None:
        raw_raw = pd.to_numeric(
            pd.Series(raw_future_ret).replace([np.inf, -np.inf], np.nan),
            errors="coerce",
        )
        if len(y_raw) != len(raw_raw):
            raise ValueError(f"{label}: directed_y 与 raw_future_ret 长度不一致")
    else:
        raw_raw = None

    y = y_raw.dropna().astype(float)
    n = int(len(y))

    if n == 0:
        return {
            "label": label, "K": k, "n": 0, "mean_y": np.nan, "t_nw": np.nan,
            "hit_rate": np.nan, "boot_p_mean_le_0": np.nan, "mean_raw": np.nan,
            "q05_y": np.nan, "q95_y": np.nan,
        }

    if raw_raw is not None:
        raw_aligned = raw_raw.loc[y.index]
        if raw_aligned.isna().any():
            raise ValueError(f"{label}: directed_y 有效样本对应的 raw_future_ret 存在缺失")
        raw = raw_aligned.astype(float)
    else:
        raw = pd.Series(dtype=float)

    _, t_nw = nw_tstat(y)

    return {
        "label": label,
        "K": k,
        "n": n,
        "mean_y": float(y.mean()),
        "t_nw": float(t_nw) if pd.notna(t_nw) else np.nan,
        "hit_rate": float((y > 0).mean()),
        "boot_p_mean_le_0": block_bootstrap_p_positive(y),
        "mean_raw": float(raw.mean()) if len(raw) else np.nan,
        "q05_y": float(y.quantile(0.05)),
        "q95_y": float(y.quantile(0.95)),
    }

def add_model_flags(scores: pd.DataFrame) -> pd.DataFrame:
    if scores.empty:
        return scores
    out = scores.copy()
    out["candidate_model"] = (
        (out["n"] >= MIN_MODEL_N)
        & (out["mean_y"] > 0)
        & (out["t_nw"] >= MIN_T_CAND)
        & (out["hit_rate"] > MIN_HIT_RATE)
    )
    out["confirmed_model"] = (
        out["candidate_model"]
        & (out["n"] >= MIN_CONFIRMED_N)
        & (out["t_nw"] >= MIN_T_CONF)
        & (out["hit_rate"] > MIN_HIT_RATE)
        & ((out["boot_p_mean_le_0"].isna()) | (out["boot_p_mean_le_0"] <= 0.10))
    )
    out["model_quality"] = np.select(
        [out["confirmed_model"], out["candidate_model"]],
        ["CONFIRMED", "CANDIDATE"],
        default="DIAGNOSTIC_ONLY",
    )
    return out


def zscore_by_commodity(df: pd.DataFrame, col: str) -> pd.Series:
    def _z(s):
        std = s.std(ddof=0)
        return (s - s.mean()) / std if pd.notna(std) and std > 0 else pd.Series(np.nan, index=s.index)
    return df.groupby("commodity")[col].transform(_z)


def pick_best_model(scores: pd.DataFrame) -> pd.DataFrame:
    if scores.empty:
        return pd.DataFrame()
    out = scores.copy()
    for c in ["confirmed_model", "candidate_model"]:
        if c not in out.columns:
            out[c] = False
    out = out.sort_values(
        ["commodity", "confirmed_model", "candidate_model", "t_nw", "mean_y", "hit_rate", "n"],
        ascending=[True, False, False, False, False, False, False],
    )
    return out.groupby("commodity", as_index=False).head(1).reset_index(drop=True)


In [10]:
def rolling_trend_stats(y: np.ndarray, h: int) -> Tuple[np.ndarray, np.ndarray]:
    n = len(y)
    tvals = np.full(n, np.nan)
    r2s = np.full(n, np.nan)
    if h < 3 or n < h:
        return tvals, r2s

    x = np.arange(h, dtype=float)
    x_mean = x.mean()
    xx = x - x_mean
    ssx = float(np.dot(xx, xx))

    for i in range(h - 1, n):
        yy = y[i - h + 1:i + 1]
        if np.isnan(yy).any():
            continue
        y_mean = float(np.mean(yy))
        yc = yy - y_mean
        beta = float(np.dot(xx, yc) / ssx)
        alpha = y_mean - beta * x_mean
        fitted = alpha + beta * x
        resid = yy - fitted
        sse = float(np.dot(resid, resid))
        sst = float(np.dot(yc, yc))
        r2s[i] = 1.0 - sse / sst if sst > 0 else np.nan
        if h > 2:
            se_beta = math.sqrt(max(sse / (h - 2), 0) / ssx) if ssx > 0 else np.nan
            tvals[i] = beta / se_beta if pd.notna(se_beta) and se_beta > 0 else np.nan
    return tvals, r2s


def add_features_one_commodity(g: pd.DataFrame) -> pd.DataFrame:
    g = g.copy().sort_values("date").reset_index(drop=True)
    g["series_age"] = np.arange(1, len(g) + 1)
    g["series_len"] = len(g)
    g["series_remain"] = g["series_len"] - g["series_age"]

    g["ret_clean"] = np.log(g["close"] / g["close"].shift(1))
    g.loc[g["series_age"] == 1, "ret_clean"] = np.nan

    prev_close = g["close"].shift(1)
    tr1 = g["high"] - g["low"]
    tr2 = (g["high"] - prev_close).abs()
    tr3 = (g["low"] - prev_close).abs()
    g["tr"] = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    g.loc[g["series_age"] == 1, "tr"] = np.nan
    g["atr20"] = g["tr"].rolling(ATR_STOP_LOOKBACK, min_periods=max(5, ATR_STOP_LOOKBACK // 2)).mean()

    g["log_close"] = np.log(g["close"])
    g["log_volume"] = np.log1p(g["volume"].clip(lower=0))
    g["sigma_roll20"] = g["ret_clean"].rolling(
        VOL_BASE_LOOKBACK,
        min_periods=max(5, VOL_BASE_LOOKBACK // 2),
    ).std(ddof=0)
    g["sigma_ewma"] = g["ret_clean"].ewm(
        alpha=1 - EWMA_LAMBDA,
        adjust=False,
        min_periods=5,
    ).std()
    g["sigma_daily_base"] = g["sigma_roll20"].combine_first(g["sigma_ewma"])
    g.loc[g["sigma_daily_base"] < MIN_SIGMA, "sigma_daily_base"] = np.nan

    g["pk_var_1"] = (np.log(g["high"] / g["low"]) ** 2) / (4 * np.log(2))
    g["pk_sigma_1"] = np.sqrt(g["pk_var_1"]).replace([np.inf, -np.inf], np.nan)

    g["std_ret_1"] = g["ret_clean"] / g["sigma_daily_base"]
    g["shock_flag"] = (g["std_ret_1"].abs() >= SHOCK_Z_THRESHOLD).astype(float)

    for h in sorted(set(H_RET_LIST + J_LIST)):
        g[f"R_{h}"] = g["ret_clean"].rolling(h, min_periods=h).sum()
        g[f"sigma_base_{h}"] = g["sigma_daily_base"] * math.sqrt(h)
        g[f"D_{h}"] = g[f"R_{h}"] / g[f"sigma_base_{h}"]
        g[f"VolZ_{h}"] = (
            (g["log_volume"] - g["log_volume"].rolling(h, min_periods=h).mean())
            / g["log_volume"].rolling(h, min_periods=h).std(ddof=0)
        )

        past_std = g["std_ret_1"].shift(1)
        g[f"RAMOM_{h}"] = past_std.rolling(h, min_periods=h).sum()
        past_ret = g["ret_clean"].shift(1)

        def _mcs(x):
            total = np.nansum(x)
            side = np.sign(total)
            if side == 0 or np.isnan(side):
                return np.nan
            return float(np.mean(np.sign(x) == side))

        g[f"MCS_{h}"] = past_ret.rolling(h, min_periods=h).apply(_mcs, raw=True)

        # 成交量只确认放量，不提供方向。
        g[f"VolumeConfirm_{h}"] = (
            g[f"RAMOM_{h}"].notna()
            & (g[f"RAMOM_{h}"] != 0)
            & (g[f"VolZ_{h}"] >= VOLUME_CONFIRM_Z)
        ).astype(float)

    y = g["log_close"].to_numpy(dtype=float)
    for h in H_TREND_LIST:
        tvals, r2s = rolling_trend_stats(y, h)
        g[f"TTrend_{h}"] = tvals
        g[f"TrendFit_{h}"] = r2s
        g[f"roll_max_close_{h}"] = g["close"].rolling(h, min_periods=max(5, h // 2)).max()
        g[f"roll_min_close_{h}"] = g["close"].rolling(h, min_periods=max(5, h // 2)).min()

    for k in K_LIST:
        g[f"fwd_ret_{k}"] = np.log(g["close"].shift(-k) / g["close"])
        g[f"valid_target_{k}"] = (g["series_remain"] >= k).astype(float)
        g.loc[g[f"valid_target_{k}"] != 1, f"fwd_ret_{k}"] = np.nan
        g[f"Y_{k}"] = g[f"fwd_ret_{k}"] / (g["sigma_daily_base"] * math.sqrt(k))

    return g.replace([np.inf, -np.inf], np.nan)


feature_panel = pd.concat(
    [add_features_one_commodity(g) for _, g in panel.groupby("commodity", sort=False)],
    ignore_index=True,
).reset_index(drop=True)

display(feature_panel.head())
display(feature_panel[["commodity", "date", "close", "ret_clean", "sigma_daily_base", "Y_5"]].tail())

,date,datetime,commodity,commodity_name,main_symbol,data_source,open,high,low,close,volume,csv_path,ohlcv_bad,series_age,series_len,series_remain,series_start,series_end,data_reliability,ret_clean,tr,atr20,log_close,log_volume,sigma_roll20,sigma_ewma,sigma_daily_base,pk_var_1,pk_sigma_1,std_ret_1,shock_flag,R_1,sigma_base_1,D_1,VolZ_1,RAMOM_1,MCS_1,VolumeConfirm_1,R_2,sigma_base_2,D_2,VolZ_2,RAMOM_2,MCS_2,VolumeConfirm_2,R_3,sigma_base_3,D_3,VolZ_3,RAMOM_3,MCS_3,VolumeConfirm_3,R_5,sigma_base_5,D_5,VolZ_5,RAMOM_5,MCS_5,VolumeConfirm_5,R_10,sigma_base_10,D_10,VolZ_10,RAMOM_10,MCS_10,VolumeConfirm_10,R_20,sigma_base_20,D_20,VolZ_20,RAMOM_20,MCS_20,VolumeConfirm_20,R_40,sigma_base_40,D_40,VolZ_40,RAMOM_40,MCS_40,VolumeConfirm_40,R_60,sigma_base_60,D_60,VolZ_60,RAMOM_60,MCS_60,VolumeConfirm_60,TTrend_10,TrendFit_10,roll_max_close_10,roll_min_close_10,TTrend_20,TrendFit_20,roll_max_close_20,roll_min_close_20,TTrend_40,TrendFit_40,roll_max_close_40,roll_min_close_40,TTrend_60,TrendFit_60,roll_max_close_60,roll_min_close_60,fwd_ret_1,valid_target_1,Y_1,fwd_ret_2,valid_target_2,Y_2,fwd_ret_3,valid_target_3,Y_3,fwd_ret_5,valid_target_5,Y_5,fwd_ret_10,valid_target_10,Y_10,fwd_ret_20,valid_target_20,Y_20
0,2016-01-05,2016-01-05,AG,沪银,KQ.m@SHFE.ag,TQ_MAIN_CONTINUOUS_KLINE_DAILY,3306.0,3330.0,3294.0,3313.0,287111.0,main_continuous_daily_trend_momentum_reversal_...,0.0,1,2544,2543,2016-01-05,2026-06-29,MAIN_CONTINUOUS_DIAGNOSTIC,NaN,NaN,NaN,8.105609,12.567628,NaN,NaN,NaN,0.000043,0.006528,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000302,1.0,NaN,0.013194,1.0,NaN,0.012896,1.0,NaN,-0.007575,1.0,NaN,0.001207,1.0,NaN,0.007518,1.0,NaN
1,2016-01-06,2016-01-06,AG,沪银,KQ.m@SHFE.ag,TQ_MAIN_CONTINUOUS_KLINE_DAILY,3314.0,3321.0,3296.0,3314.0,204526.0,main_continuous_daily_trend_momentum_reversal_...,0.0,2,2544,2542,2016-01-05,2026-06-29,MAIN_CONTINUOUS_DIAGNOSTIC,0.000302,25.0,NaN,8.105911,12.228455,NaN,NaN,NaN,0.000021,0.004538,NaN,0.0,0.000302,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,-1.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.012892,1.0,NaN,0.012594,1.0,NaN,0.010506,1.0,NaN,-0.012143,1.0,NaN,0.000302,1.0,NaN,0.008114,1.0,NaN
2,2016-01-07,2016-01-07,AG,沪银,KQ.m@SHFE.ag,TQ_MAIN_CONTINUOUS_KLINE_DAILY,3322.0,3385.0,3308.0,3357.0,488186.0,main_continuous_daily_trend_momentum_reversal_...,0.0,3,2544,2541,2016-01-05,2026-06-29,MAIN_CONTINUOUS_DIAGNOSTIC,0.012892,77.0,NaN,8.118803,13.098454,NaN,NaN,NaN,0.000191,0.013819,NaN,0.0,0.012892,NaN,NaN,NaN,NaN,1.0,0.0,0.013194,NaN,NaN,1.0,NaN,NaN,0.0,NaN,NaN,NaN,1.304173,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.000298,1.0,NaN,-0.002386,1.0,NaN,-0.020768,1.0,NaN,-0.013495,1.0,NaN,-0.007775,1.0,NaN,0.009192,1.0,NaN
3,2016-01-08,2016-01-08,AG,沪银,KQ.m@SHFE.ag,TQ_MAIN_CONTINUOUS_KLINE_DAILY,3367.0,3390.0,3345.0,3356.0,379737.0,main_continuous_daily_trend_momentum_reversal_...,0.0,4,2544,2540,2016-01-05,2026-06-29,MAIN_CONTINUOUS_DIAGNOSTIC,-0.000298,45.0,NaN,8.118505,12.847237,NaN,NaN,NaN,0.000064,0.008025,NaN,0.0,-0.000298,NaN,NaN,NaN,NaN,1.0,0.0,0.012594,NaN,NaN,-1.0,NaN,1.0,0.0,0.012896,NaN,NaN,0.335135,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.002088,1.0,NaN,-0.020470,1.0,NaN,-0.024737,1.0,NaN,-0.018647,1.0,NaN,-0.00988

,commodity,date,close,ret_clean,sigma_daily_base,Y_5
24186,V,2026-06-23,4444.0,-0.020270,0.010201,NaN
24187,V,2026-06-24,4490.0,0.010298,0.010578,NaN
24188,V,2026-06-25,4469.0,-0.004688,0.010426,NaN
24189,V,2026-06-26,4388.0,-0.018291,0.010006,NaN
24190,V,2026-06-29,4391.0,0.000683,0.010092,NaN


In [11]:
def feature_target_audit(df: pd.DataFrame) -> Dict[str, Dict]:
    report = {}

    ret_bad = {}
    for h in H_RET_LIST:
        ret_bad[h] = int(df.loc[df["series_age"] < h + 1, f"R_{h}"].notna().sum())
    report["return_feature_non_nan_too_early"] = ret_bad

    trend_bad = {}
    for h in H_TREND_LIST:
        trend_bad[h] = int(df.loc[df["series_age"] < h, f"TTrend_{h}"].notna().sum())
    report["trend_feature_non_nan_too_early"] = trend_bad

    target_bad = {}
    for k in K_LIST:
        target_bad[k] = int(df.loc[df["series_remain"] < k, f"fwd_ret_{k}"].notna().sum())
    report["future_target_non_nan_beyond_series_end"] = target_bad

    bad_ohlc = int((
        (df["open"] <= 0)
        | (df["high"] <= 0)
        | (df["low"] <= 0)
        | (df["close"] <= 0)
        | (df["high"] < df[["open", "close", "low"]].max(axis=1))
        | (df["low"] > df[["open", "close", "high"]].min(axis=1))
    ).sum())
    report["bad_ohlc_after_cleaning"] = {"count": bad_ohlc}

    return report


audit_feature_target = feature_target_audit(feature_panel)
display(audit_feature_target)

audit_failed = any(
    count != 0
    for group in audit_feature_target.values()
    for count in group.values()
)

if audit_failed:
    raise RuntimeError(f"Feature / target audit failed: {audit_feature_target}")

{'return_feature_non_nan_too_early': {1: 0,
  2: 0,
  3: 0,
  5: 0,
  10: 0,
  20: 0,
  40: 0,
  60: 0},
 'trend_feature_non_nan_too_early': {10: 0, 20: 0, 40: 0, 60: 0},
 'future_target_non_nan_beyond_series_end': {1: 0,
  2: 0,
  3: 0,
  5: 0,
  10: 0,
  20: 0},
 'bad_ohlc_after_cleaning': {'count': 0}}

In [12]:
def trend_score_table(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for commodity, g in df.groupby("commodity"):
        for h in H_TREND_LIST:
            side = np.sign(g[f"TTrend_{h}"])
            trend_exists = (g[f"TTrend_{h}"].abs() >= TREND_T_THRESHOLD) & (g[f"TrendFit_{h}"] >= TREND_FIT_THRESHOLD)
            for k in K_LIST:
                mask = trend_exists & (g[f"valid_target_{k}"] == 1) & g[f"Y_{k}"].notna() & side.notna() & (side != 0)
                if mask.sum() == 0:
                    continue
                stat = summarize_directed_returns(
                    side.loc[mask] * g.loc[mask, f"Y_{k}"],
                    raw_future_ret=side.loc[mask] * g.loc[mask, f"fwd_ret_{k}"],
                    k=k,
                    label=f"{commodity}_Trend_h{h}_K{k}",
                )
                stat.update({"commodity": commodity, "h": h, "signal_type": "Trend"})
                rows.append(stat)

    out = pd.DataFrame(rows)
    if out.empty:
        return out
    out = add_model_flags(out)
    return out.sort_values(["commodity", "confirmed_model", "candidate_model", "t_nw"], ascending=[True, False, False, False])


trend_scores = trend_score_table(feature_panel)
trend_best = pick_best_model(trend_scores)
display(trend_scores.head(12))
display(trend_best)
trend_scores.to_csv(RESULT_DIR / "trend_model_scores.csv", index=False)


,label,K,n,mean_y,t_nw,hit_rate,boot_p_mean_le_0,mean_raw,q05_y,q95_y,commodity,h,signal_type,candidate_model,confirmed_model,model_quality
17,AG_Trend_h40_K20,20,1697,0.243651,3.107824,0.506777,0.0000,0.012075,-1.268326,2.738127,AG,40,Trend,True,True,CONFIRMED
16,AG_Trend_h40_K10,10,1701,0.151802,2.226763,0.507937,0.0000,0.006583,-1.539569,2.347898,AG,40,Trend,True,True,CONFIRMED
15,AG_Trend_h40_K5,5,1706,0.115528,2.001000,0.504689,0.0125,0.003856,-1.651821,2.174599,AG,40,Trend,True,True,CONFIRMED
14,AG_Trend_h40_K3,3,1708,0.092275,1.940514,0.503513,0.0175,0.002494,-1.724347,2.218203,AG,40,Trend,True,False,CANDIDATE
13,AG_Trend_h40_K2,2,1709,0.076848,1.936709,0.503803,0.0250,0.001715,-1.703829,2.086099,AG,40,Trend,True,False,CANDIDATE
22,AG_Trend_h60_K10,10,1693,0.116147,1.561386,0.505611,0.0425,0.003211,-1.775047,2.527431,AG,60,Trend,True,False,CANDIDATE
21,AG_Trend_h60_K5,5,1693,0.076119,1.231406,0.505021,0.1000,0.001424,-1.922794,2.270896,AG,60,Trend,True,False,CANDIDATE
11,AG_Trend_h20_K20,20,1663,0.154465,2.051958,0.482261,0.0025,0.009591,-1.390773,2.440514,AG,20,Trend,False,False,DIAGNOSTIC_ONLY
12,AG_Trend_h40_K1,1,1710,0.052874,1.811622,0.497076,0.0325,0.000880,-1.746457,2.145128,AG,40,Trend,False,False,DIAGNOSTIC_ONLY
23,AG_Trend_h60_K20,20,1693,0.144847,1.601069,0.464265,0.0325,0.005081,-1.516340,2.758278,AG,60,Trend,False,False,DIAGNOSTIC_ONLY


,label,K,n,mean_y,t_nw,hit_rate,boot_p_mean_le_0,mean_raw,q05_y,q95_y,commodity,h,signal_type,candidate_model,confirmed_model,model_quality
0,AG_Trend_h40_K20,20,1697,0.243651,3.107824,0.506777,0.0000,0.012075,-1.268326,2.738127,AG,40,Trend,True,True,CONFIRMED
1,CF_Trend_h40_K10,10,1766,0.139312,1.790203,0.520385,0.0125,0.002698,-1.638308,2.205744,CF,40,Trend,True,False,CANDIDATE
2,EB_Trend_h10_K2,2,1015,0.108912,2.079637,0.525123,0.0150,0.002190,-1.716068,2.194631,EB,10,Trend,True,True,CONFIRMED
3,FG_Trend_h10_K20,20,1550,0.115682,1.719143,0.519355,0.0175,0.003987,-1.852035,2.046783,FG,10,Trend,True,False,CANDIDATE
4,LH_Trend_h40_K20,20,940,0.304104,2.128568,0.550000,0.0100,0.017679,-2.217295,3.389378,LH,40,Trend,True,True,CONFIRMED
5,M_Trend_h20_K10,10,1814,0.054202,0.867564,0.493385,0.1800,0.001896,-1.793508,2.131066,M,20,Trend,False,False,DIAGNOSTIC_ONLY
6,MA_Trend_h40_K20,20,1725,0.179631,2.517070,0.513043,0.0000,0.006951,-1.503839,2.208188,MA,40,Trend,True,True,CONFIRMED
7,SA_Trend_h20_K10,10,1120,0.131157,1.367588,0.518750,0.0525,0.004917,-2.027490,2.608972,SA,20,Trend,True,False,CANDIDATE
8,SP_Trend_h10_K20,20,1095,0.153981,1.867982,0.521461,0.0125,0.005870,-1.831228,2.498315,SP,10,Trend,True,False,CANDIDATE
9,SR_Trend_h60_K20,20,1613,0.118699,1.768501,0.507130,0.0125,0.001629,-1.465224,2.056469,SR,60,Trend,True,False,CANDIDATE


In [13]:
TREND_ESTABLISH_CONFIRM_DAYS = globals().get("TREND_ESTABLISH_CONFIRM_DAYS", 2)
TREND_OVEREXT_MIN_PERIODS = globals().get("TREND_OVEREXT_MIN_PERIODS", 120)


def trend_level_label(h: float) -> str:
    if pd.isna(h):
        return "NONE"
    h = int(h)
    if h <= 10:
        return "SHORT"
    if h <= 40:
        return "MEDIUM"
    return "LONG"


def add_trend_state_columns(df: pd.DataFrame, best: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    for col in [
        "h_trend_star", "K_trend_eval", "trend_level_label", "trend_confirmed_model",
        "trend_candidate_model", "trend_model_scope", "TrendMainT", "TrendMainFit",
        "TrendMainSide", "TrendMainExists", "TrendEstablished", "TrendStart",
        "TrendOverextendedThr", "TrendOverextended", "TrendEnd_sig",
        "TrendEnd_flip", "TrendEnd_ATR", "TrendEnd_critical",
        "TrendEndRisk", "TrendEndConfirmed", "TrendEnd",
    ]:
        out[col] = np.nan

    out["trend_level_label"] = pd.Series(pd.NA, index=out.index, dtype="object")
    out["trend_model_scope"] = pd.Series("NO_MODEL", index=out.index, dtype="object")
    out["trend_confirmed_model"] = False
    out["trend_candidate_model"] = False

    for _, row in best.iterrows():
        commodity = row["commodity"]
        h = int(row["h"])
        k = int(row["K"])
        m = out["commodity"] == commodity

        out.loc[m, "h_trend_star"] = h
        out.loc[m, "K_trend_eval"] = k
        out.loc[m, "trend_level_label"] = trend_level_label(h)
        out.loc[m, "trend_confirmed_model"] = bool(row.get("confirmed_model", False))
        out.loc[m, "trend_candidate_model"] = bool(row.get("candidate_model", False))
        out.loc[m, "trend_model_scope"] = row.get("model_quality", "DIAGNOSTIC_ONLY")

        out.loc[m, "TrendMainT"] = out.loc[m, f"TTrend_{h}"]
        out.loc[m, "TrendMainFit"] = out.loc[m, f"TrendFit_{h}"]
        out.loc[m, "TrendMainSide"] = np.sign(out.loc[m, "TrendMainT"])

        out.loc[m, "TrendMainExists"] = (
            (out.loc[m, "TrendMainT"].abs() >= TREND_T_THRESHOLD)
            & (out.loc[m, "TrendMainFit"] >= TREND_FIT_THRESHOLD)
        ).astype(float)

        abs_t = out.loc[m, "TrendMainT"].abs()
        hist_q = (
            abs_t
            .expanding(min_periods=TREND_OVEREXT_MIN_PERIODS)
            .quantile(TREND_OVEREXT_Q)
            .shift(1)
        )
        fixed_thr = TREND_T_THRESHOLD + 1.0
        over_thr = hist_q.clip(lower=fixed_thr).fillna(fixed_thr)

        out.loc[m, "TrendOverextendedThr"] = over_thr
        out.loc[m, "TrendOverextended"] = (
            (out.loc[m, "TrendMainExists"] == 1)
            & (out.loc[m, "TrendMainT"].abs() >= over_thr)
        ).astype(float)

        up_stop = out.loc[m, "close"] < (
            out.loc[m, f"roll_max_close_{h}"] - ATR_STOP_MULT * out.loc[m, "atr20"]
        )
        dn_stop = out.loc[m, "close"] > (
            out.loc[m, f"roll_min_close_{h}"] + ATR_STOP_MULT * out.loc[m, "atr20"]
        )

        out.loc[m, "TrendEnd_ATR"] = (
            ((out.loc[m, "TrendMainSide"] > 0) & up_stop)
            | ((out.loc[m, "TrendMainSide"] < 0) & dn_stop)
        ).astype(float)

    frames = []
    for _, g in out.groupby("commodity", sort=False):
        g = g.copy().sort_values("date").reset_index(drop=True)

        active = (
            (g["TrendMainExists"] == 1)
            & g["TrendMainSide"].notna()
            & (g["TrendMainSide"] != 0)
        )
        side_key = g["TrendMainSide"].where(active, 0)

        block_id = (side_key != side_key.shift(1)).cumsum()
        active_run_len = active.astype(int).groupby(block_id).cumsum()

        g["TrendEstablished"] = (
            active & (active_run_len >= TREND_ESTABLISH_CONFIRM_DAYS)
        ).astype(float)

        g["TrendStart"] = (
            (g["TrendEstablished"] == 1)
            & (g["TrendEstablished"].shift(1) != 1)
        ).astype(float)

        prev_established = g["TrendEstablished"].shift(1) == 1
        prev_side = g["TrendMainSide"].shift(1)

        g["TrendEnd_sig"] = (
            prev_established
            & (g["TrendMainExists"] != 1)
        ).astype(float)

        g["TrendEnd_flip"] = (
            prev_established
            & (g["TrendMainExists"] == 1)
            & g["TrendMainSide"].notna()
            & prev_side.notna()
            & (np.sign(g["TrendMainSide"]) == -np.sign(prev_side))
        ).astype(float)

        raw_atr_end = g["TrendEnd_ATR"].fillna(0) == 1
        g["TrendEnd_ATR"] = (
            prev_established
            & raw_atr_end
        ).astype(float)

        g["TrendEndRisk"] = (
            (g["TrendEstablished"] == 1)
            & (g["TrendOverextended"] == 1)
        ).astype(float)

        g["TrendEnd_critical"] = g["TrendEndRisk"]

        g["TrendEndConfirmed"] = (
            (g["TrendEnd_sig"] == 1)
            | (g["TrendEnd_flip"] == 1)
            | (g["TrendEnd_ATR"] == 1)
        ).astype(float)

        # TrendEnd 仅表示确认结束；结束风险看 TrendEndRisk。
        g["TrendEnd"] = g["TrendEndConfirmed"]

        frames.append(g)

    return pd.concat(frames, ignore_index=True).replace([np.inf, -np.inf], np.nan)


panel_state = add_trend_state_columns(feature_panel, trend_best)

display(panel_state[[
    "date", "commodity", "h_trend_star", "K_trend_eval", "trend_model_scope",
    "TrendMainT", "TrendMainFit", "TrendMainExists", "TrendEstablished",
    "TrendOverextended", "TrendEndRisk", "TrendEndConfirmed", "TrendEnd"
]].tail(24))

,date,commodity,h_trend_star,K_trend_eval,trend_model_scope,TrendMainT,TrendMainFit,TrendMainExists,TrendEstablished,TrendOverextended,TrendEndRisk,TrendEndConfirmed,TrendEnd
24167,2026-05-26,V,20.0,20.0,CANDIDATE,-8.065806,0.783282,1.0,1.0,0.0,0.0,0.0,0.0
24168,2026-05-27,V,20.0,20.0,CANDIDATE,-11.041636,0.871353,1.0,1.0,1.0,1.0,0.0,0.0
24169,2026-05-28,V,20.0,20.0,CANDIDATE,-10.948902,0.869450,1.0,1.0,1.0,1.0,0.0,0.0
24170,2026-05-29,V,20.0,20.0,CANDIDATE,-9.322858,0.828434,1.0,1.0,0.0,0.0,0.0,0.0
24171,2026-06-01,V,20.0,20.0,CANDIDATE,-6.973966,0.729877,1.0,1.0,0.0,0.0,0.0,0.0
24172,2026-06-02,V,20.0,20.0,CANDIDATE,-5.854191,0.655644,1.0,1.0,0.0,0.0,0.0,0.0
24173,2026-06-03,V,20.0,20.0,CANDIDATE,-4.932161,0.574732,1.0,1.0,0.0,0.0,0.0,0.0
24174,2026-06-04,V,20.0,20.0,CANDIDATE,-4.973897,0.578845,1.0,1.0,0.0,0.0,0.0,0.0
24175,2026-06-05,V,20.0,20.0,CANDIDATE,-5.121010,0.592988,1.0,1.0,0.0,0.0,0.0,0.0
24176,2026-06-08,V,20.0,20.0,CANDIDATE,-5.156949,0.596359,1.0,1.0,0.0,0.0,0.0,0.0


In [14]:

SURVIVAL_HORIZONS = sorted(set(globals().get("SURVIVAL_HORIZONS", K_LIST)))
TREND_DURATION_MIN_COMPLETED = globals().get("TREND_DURATION_MIN_COMPLETED", 5)


def _trend_side(x) -> int:
    if pd.isna(x):
        return 0
    s = int(np.sign(x))
    return s if s in (-1, 1) else 0


def _trend_side_label(side: int) -> str:
    return "UP" if side > 0 else "DOWN" if side < 0 else "NONE"


def _trend_end_reason(row: pd.Series) -> str:
    reasons = []
    if row.get("TrendEnd_sig", 0) == 1:
        reasons.append("SIG_LOST")
    if row.get("TrendEnd_flip", 0) == 1:
        reasons.append("SIDE_FLIP")
    if row.get("TrendEnd_ATR", 0) == 1:
        reasons.append("ATR_STOP")
    return "|".join(reasons) if reasons else "STATE_LOST"


def build_trend_episodes(state: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    required = [
        "date", "commodity", "series_remain",
        "TrendMainT", "TrendMainFit", "TrendMainSide",
        "TrendMainExists", "TrendEstablished",
        "TrendEndRisk", "TrendEndConfirmed",
        "h_trend_star", "K_trend_eval", "trend_model_scope",
    ]
    missing = [c for c in required if c not in state.columns]
    if missing:
        raise KeyError(f"Missing required columns: {missing}")

    out = state.copy().sort_values(["commodity", "date"]).reset_index(drop=True)

    num_cols = [
        "TrendEpisodeSide", "TrendAge", "TrendDaysToEnd",
        "TrendRemainingLife", "TrendEpisodeCensored",
    ]
    obj_cols = [
        "TrendEpisodeID", "TrendEpisodeSideLabel",
        "TrendEpisodeStartDate", "TrendEpisodeRawStartDate",
        "TrendEpisodeEndDate", "TrendEpisodeEndReason",
    ]

    for c in num_cols:
        out[c] = np.nan
    for c in obj_cols:
        out[c] = pd.Series(pd.NA, index=out.index, dtype="object")

    episodes = []

    def raw_start_index(g: pd.DataFrame, idxs: list, pos: int, side: int):
        p = pos
        while p > 0:
            prev_idx = idxs[p - 1]
            prev_exists = g.loc[prev_idx, "TrendMainExists"] == 1
            prev_side = _trend_side(g.loc[prev_idx, "TrendMainSide"])
            if prev_exists and prev_side == side:
                p -= 1
            else:
                break
        return idxs[p]

    def close_episode(g: pd.DataFrame, cur: dict, end_idx=None, end_pos=None, reason="CENSORED"):
        idxs_active = cur["idxs"]
        poss_active = cur["poss"]
        if not idxs_active:
            return

        ended = end_idx is not None
        end_date = g.loc[end_idx, "date"] if ended else pd.NaT
        ep_slice = out.loc[idxs_active]
        risk_dates = ep_slice.loc[ep_slice["TrendEndRisk"] == 1, "date"]

        for age, (pos_i, idx_i) in enumerate(zip(poss_active, idxs_active), start=1):
            out.loc[idx_i, "TrendEpisodeID"] = cur["episode_id"]
            out.loc[idx_i, "TrendEpisodeSide"] = cur["side"]
            out.loc[idx_i, "TrendEpisodeSideLabel"] = _trend_side_label(cur["side"])
            out.loc[idx_i, "TrendEpisodeStartDate"] = g.loc[cur["start_idx"], "date"]
            out.loc[idx_i, "TrendEpisodeRawStartDate"] = g.loc[cur["raw_start_idx"], "date"]
            out.loc[idx_i, "TrendEpisodeEndDate"] = end_date
            out.loc[idx_i, "TrendEpisodeEndReason"] = reason
            out.loc[idx_i, "TrendAge"] = age
            out.loc[idx_i, "TrendEpisodeCensored"] = 0 if ended else 1

            if ended:
                days_left = int(end_pos - pos_i)
                out.loc[idx_i, "TrendDaysToEnd"] = days_left
                out.loc[idx_i, "TrendRemainingLife"] = days_left

        episodes.append({
            "commodity": cur["commodity"],
            "trend_episode_id": cur["episode_id"],
            "trend_side": cur["side"],
            "trend_side_label": _trend_side_label(cur["side"]),
            "h_trend_star": out.loc[cur["start_idx"], "h_trend_star"],
            "K_trend_eval": out.loc[cur["start_idx"], "K_trend_eval"],
            "trend_model_scope": out.loc[cur["start_idx"], "trend_model_scope"],
            "raw_start_date": g.loc[cur["raw_start_idx"], "date"],
            "established_date": g.loc[cur["start_idx"], "date"],
            "last_established_date": g.loc[idxs_active[-1], "date"],
            "end_signal_date": end_date,
            "end_reason": reason,
            "active_duration_days": len(idxs_active),
            "duration_to_end_signal_days": int(end_pos - cur["start_pos"]) if ended else np.nan,
            "is_censored": not ended,
            "ever_overextended": bool((ep_slice["TrendEndRisk"] == 1).any()),
            "first_overextended_date": risk_dates.iloc[0] if len(risk_dates) else pd.NaT,
            "start_T": out.loc[cur["start_idx"], "TrendMainT"],
            "start_fit": out.loc[cur["start_idx"], "TrendMainFit"],
            "max_abs_T": ep_slice["TrendMainT"].abs().max(),
            "mean_fit": ep_slice["TrendMainFit"].mean(),
            "end_T": g.loc[end_idx, "TrendMainT"] if ended else np.nan,
            "end_fit": g.loc[end_idx, "TrendMainFit"] if ended else np.nan,
        })

    for commodity, g in out.groupby("commodity", sort=False):
        g = g.sort_values("date")
        idxs = list(g.index)
        cur = None
        ep_num = 0

        for pos, idx in enumerate(idxs):
            row = g.loc[idx]
            side = _trend_side(row["TrendMainSide"])
            established = (row["TrendEstablished"] == 1) and side != 0
            end_confirmed = row["TrendEndConfirmed"] == 1

            if cur is None:
                if established:
                    ep_num += 1
                    cur = {
                        "commodity": commodity,
                        "episode_id": f"{commodity}_TREND_{ep_num:04d}",
                        "side": side,
                        "start_pos": pos,
                        "start_idx": idx,
                        "raw_start_idx": raw_start_index(g, idxs, pos, side),
                        "poss": [pos],
                        "idxs": [idx],
                    }
                    if end_confirmed:
                        close_episode(g, cur, idx, pos, _trend_end_reason(row))
                        cur = None
                continue

            same_episode = established and side == cur["side"]

            if same_episode:
                cur["poss"].append(pos)
                cur["idxs"].append(idx)

                if end_confirmed:
                    close_episode(g, cur, idx, pos, _trend_end_reason(row))
                    cur = None
                continue

            close_episode(
                g,
                cur,
                idx,
                pos,
                _trend_end_reason(row) if end_confirmed else "STATE_LOST_UNFLAGGED",
            )
            cur = None

            if established:
                ep_num += 1
                cur = {
                    "commodity": commodity,
                    "episode_id": f"{commodity}_TREND_{ep_num:04d}",
                    "side": side,
                    "start_pos": pos,
                    "start_idx": idx,
                    "raw_start_idx": raw_start_index(g, idxs, pos, side),
                    "poss": [pos],
                    "idxs": [idx],
                }
                if end_confirmed:
                    close_episode(g, cur, idx, pos, _trend_end_reason(row))
                    cur = None

        if cur is not None:
            close_episode(g, cur)

    return out.replace([np.inf, -np.inf], np.nan), pd.DataFrame(episodes).replace([np.inf, -np.inf], np.nan)


def summarize_trend_durations(episodes: pd.DataFrame) -> pd.DataFrame:
    if episodes.empty:
        return pd.DataFrame()

    rows = []

    for keys, g in episodes.groupby(["commodity", "trend_side_label"], sort=False):
        commodity, side = keys
        ended = g.loc[~g["is_censored"]]

        rows.append({
            "commodity": commodity,
            "trend_side_label": side,
            "n_episodes": len(g),
            "n_ended": len(ended),
            "n_censored": int(g["is_censored"].sum()),
            "overextended_rate": float(g["ever_overextended"].mean()),
            "mean_active_duration": float(g["active_duration_days"].mean()),
            "median_active_duration": float(g["active_duration_days"].median()),
            "q25_active_duration": float(g["active_duration_days"].quantile(0.25)),
            "q75_active_duration": float(g["active_duration_days"].quantile(0.75)),
            "mean_duration_to_end_signal": float(ended["duration_to_end_signal_days"].mean()) if len(ended) else np.nan,
            "median_duration_to_end_signal": float(ended["duration_to_end_signal_days"].median()) if len(ended) else np.nan,
        })

    return pd.DataFrame(rows)


def trend_survival_by_horizon(panel_duration: pd.DataFrame, horizons=None) -> pd.DataFrame:
    horizons = SURVIVAL_HORIZONS if horizons is None else horizons

    active = panel_duration.loc[
        panel_duration["TrendEpisodeID"].notna()
        & (panel_duration["TrendEstablished"] == 1)
    ].copy()

    if active.empty:
        return pd.DataFrame()

    rows = []

    for (commodity, side, risk), g in active.assign(
        risk_state=np.where(active["TrendEndRisk"] == 1, "RISK", "NO_RISK")
    ).groupby(["commodity", "TrendEpisodeSideLabel", "risk_state"], sort=False):

        for h in horizons:
            known = g["TrendDaysToEnd"].notna() | (g["series_remain"] >= h)
            sample = g.loc[known]
            n = len(sample)

            if n == 0:
                p_end = np.nan
                n_end = 0
            else:
                end_within = sample["TrendDaysToEnd"].notna() & (sample["TrendDaysToEnd"] <= h)
                n_end = int(end_within.sum())
                p_end = float(n_end / n)

            rows.append({
                "commodity": commodity,
                "trend_side_label": side,
                "risk_state": risk,
                "horizon": h,
                "n_state_obs": n,
                "n_end_within_h": n_end,
                "p_end_within_h": p_end,
                "p_survive_beyond_h": np.nan if pd.isna(p_end) else 1.0 - p_end,
            })

    return pd.DataFrame(rows)


def current_trend_life_report(
    panel_duration: pd.DataFrame,
    episodes: pd.DataFrame,
    horizons=None,
    min_completed: int = TREND_DURATION_MIN_COMPLETED,
) -> pd.DataFrame:
    horizons = SURVIVAL_HORIZONS if horizons is None else horizons

    latest = (
        panel_duration
        .sort_values(["commodity", "date"])
        .groupby("commodity", sort=False)
        .tail(1)
        .copy()
    )

    completed = episodes.loc[~episodes["is_censored"]].copy()
    rows = []

    for _, row in latest.iterrows():
        commodity = row["commodity"]
        side = _trend_side(row["TrendMainSide"])
        age = int(row["TrendAge"]) if pd.notna(row["TrendAge"]) else np.nan
        established_now = (row["TrendEstablished"] == 1) and side != 0 and pd.notna(age)

        out = {
            "commodity": commodity,
            "date": row["date"],
            "trend_established_now": bool(established_now),
            "trend_side_label": _trend_side_label(side),
            "trend_age": age,
            "trend_end_risk": bool(row.get("TrendEndRisk", 0) == 1),
            "trend_end_confirmed": bool(row.get("TrendEndConfirmed", 0) == 1),
            "h_trend_star": row.get("h_trend_star", np.nan),
            "K_trend_eval": row.get("K_trend_eval", np.nan),
            "trend_model_scope": row.get("trend_model_scope", "NO_MODEL"),
        }

        if not established_now:
            comparable = pd.Series(dtype=float)
        else:
            comp = completed.loc[
                (completed["commodity"] == commodity)
                & (completed["trend_side"] == side)
                & (completed["active_duration_days"] >= age)
            ]
            comparable = comp["duration_to_end_signal_days"] - (age - 1)
            comparable = comparable.loc[comparable >= 0].astype(float)

        out["n_comparable_completed"] = int(len(comparable))
        out["mean_remaining_days"] = float(comparable.mean()) if len(comparable) >= min_completed else np.nan
        out["median_remaining_days"] = float(comparable.median()) if len(comparable) >= min_completed else np.nan

        for h in horizons:
            out[f"p_end_within_{h}d"] = (
                float((comparable <= h).mean()) if len(comparable) >= min_completed else np.nan
            )

        rows.append(out)

    return pd.DataFrame(rows)


panel_state_duration, trend_episode_table = build_trend_episodes(panel_state)
trend_duration_summary = summarize_trend_durations(trend_episode_table)
trend_survival_table = trend_survival_by_horizon(panel_state_duration)
current_trend_life_report = current_trend_life_report(panel_state_duration, trend_episode_table)

display(trend_episode_table.tail(12))
display(trend_duration_summary)
display(trend_survival_table.head(20))
display(current_trend_life_report)

trend_episode_table.to_csv(RESULT_DIR / "trend_episode_table.csv", index=False)
trend_duration_summary.to_csv(RESULT_DIR / "trend_duration_summary.csv", index=False)
trend_survival_table.to_csv(RESULT_DIR / "trend_survival_table.csv", index=False)
current_trend_life_report.to_csv(RESULT_DIR / "current_trend_life_report.csv", index=False)
panel_state_duration.to_csv(RESULT_DIR / "panel_state_duration.csv", index=False)



,commodity,trend_episode_id,trend_side,trend_side_label,h_trend_star,K_trend_eval,trend_model_scope,raw_start_date,established_date,last_established_date,end_signal_date,end_reason,active_duration_days,duration_to_end_signal_days,is_censored,ever_overextended,first_overextended_date,start_T,start_fit,max_abs_T,mean_fit,end_T,end_fit
2283,V,V_TREND_0134,-1,DOWN,20.0,20.0,CANDIDATE,2025-08-29,2025-09-01,2025-09-17,2025-09-18,SIG_LOST,13,13.0,False,False,NaT,-3.821866,0.447966,8.936780,0.592813,-2.066710,0.191785
2284,V,V_TREND_0135,1,UP,20.0,20.0,CANDIDATE,2025-09-25,2025-09-26,2025-09-26,2025-09-29,SIG_LOST,1,1.0,False,False,NaT,2.444102,0.249175,2.444102,0.249175,2.085961,0.194675
2285,V,V_TREND_0136,-1,DOWN,20.0,20.0,CANDIDATE,2025-10-13,2025-10-14,2025-10-31,2025-11-03,SIG_LOST,14,14.0,False,True,2025-10-21,-3.185136,0.360457,11.805938,0.633783,-1.887245,0.165186
2286,V,V_TREND_0137,-1,DOWN,20.0,20.0,CANDIDATE,2025-11-10,2025-11-11,2025-12-16,2025-12-16,ATR_STOP,26,25.0,False,True,2025-11-20,-3.018054,0.336005,12.901714,0.583276,-3.855903,0.452355
2287,V,V_TREND_0138,1,UP,20.0,20.0,CANDIDATE,2025-12-26,2025-12-29,2026-01-20,2026-01-21,SIG_LOST,15,15.0,False,False,NaT,3.683932,0.429863,9.033616,0.651466,1.062429,0.059008
2288,V,V_TREND_0139,1,UP,20.0,20.0,CANDIDATE,2026-02-04,2026-02-05,2026-02-13,2026-02-13,ATR_STOP,7,6.0,False,False,NaT,3.915763,0.459998,4.986547,0.505213,3.055623,0.341548
2289,V,V_TREND_0140,1,UP,20.0,20.0,CANDIDATE,2026-02-04,2026-02-24,2026-02-24,2026-02-25,SIG_LOST,1,1.0,False,False,NaT,2.318611,0.229978,2.318611,0.229978,1.681125,0.135703
2290,V,V_TREND_0141,-1,DOWN,20.0,20.0,CANDIDATE,2026-03-04,2026-03-05,2026-03-05,2026-03-06,SIG_LOST|ATR_STOP,1,1.0,False,False,NaT,-2.298317,0.226879,2.298317,0.226879,-0.899484,0.043015
2291,V,V_TREND_0142,1,UP,20.0,20.0,CANDIDATE,2026-03-11,2026-03-12,2026-03-31,2026-03-31,ATR_STOP,14,13.0,False,True,2026-03-20,3.630617,0.422732,13.164860,0.697531,2.608100,0.274257
2292,V,V_TREND_0143,-1,DOWN,20.0,20.0,CANDIDATE,2026-04-08,2026-04-09,2026-04-27,2026-04-28,SIG_LOST,13,13.0,False,False,NaT,-5.074330,0.588560,10.053496,0.677131,-1.164448,0.070053


,commodity,trend_side_label,n_episodes,n_ended,n_censored,overextended_rate,mean_active_duration,median_active_duration,q25_active_duration,q75_active_duration,mean_duration_to_end_signal,median_duration_to_end_signal
0,AG,UP,167,167,0,0.203593,5.850299,1.0,1.0,3.00,4.886228,0.0
1,AG,DOWN,177,176,1,0.090395,3.824859,1.0,1.0,2.00,2.812500,0.0
2,CF,DOWN,114,113,1,0.149123,6.210526,1.0,1.0,3.75,5.247788,0.0
3,CF,UP,133,133,0,0.142857,7.609023,1.0,1.0,6.00,6.736842,0.0
4,EB,DOWN,72,71,1,0.375000,6.222222,5.0,2.0,8.00,6.169014,5.0
5,EB,UP,61,61,0,0.442623,6.852459,5.0,3.0,9.00,6.836066,5.0
6,FG,UP,103,103,0,0.417476,6.553398,5.0,3.5,8.00,6.533981,5.0
7,FG,DOWN,101,100,1,0.376238,6.564356,5.0,3.0,9.00,6.600000,5.0
8,LH,UP,87,86,1,0.080460,4.080460,1.0,1.0,3.50,3.104651,0.0
9,LH,DOWN,99,99,0,0.373737,5.737374,1.0,1.0,3.50,4.767677,0.0


,commodity,trend_side_label,risk_state,horizon,n_state_obs,n_end_within_h,p_end_within_h,p_survive_beyond_h
0,AG,UP,RISK,1,312,35,0.112179,0.887821
1,AG,UP,RISK,2,312,47,0.150641,0.849359
2,AG,UP,RISK,3,312,59,0.189103,0.810897
3,AG,UP,RISK,5,312,81,0.259615,0.740385
4,AG,UP,RISK,10,312,134,0.429487,0.570513
5,AG,UP,RISK,20,312,224,0.717949,0.282051
6,AG,DOWN,NO_RISK,1,596,199,0.333893,0.666107
7,AG,DOWN,NO_RISK,2,595,234,0.393277,0.606723
8,AG,DOWN,NO_RISK,3,594,265,0.446128,0.553872
9,AG,DOWN,NO_RISK,5,592,313,0.528716,0.471284


,commodity,date,trend_established_now,trend_side_label,trend_age,trend_end_risk,trend_end_confirmed,h_trend_star,K_trend_eval,trend_model_scope,n_comparable_completed,mean_remaining_days,median_remaining_days,p_end_within_1d,p_end_within_2d,p_end_within_3d,p_end_within_5d,p_end_within_10d,p_end_within_20d
0,AG,2026-06-29,True,DOWN,13.0,False,False,40.0,20.0,CONFIRMED,13,15.769231,15.0,0.230769,0.230769,0.230769,0.307692,0.461538,0.692308
1,CF,2026-06-29,True,DOWN,9.0,False,False,40.0,10.0,CANDIDATE,23,16.000000,9.0,0.217391,0.217391,0.217391,0.304348,0.521739,0.739130
2,EB,2026-06-29,True,DOWN,10.0,True,False,10.0,2.0,CONFIRMED,15,5.600000,3.0,0.133333,0.333333,0.533333,0.666667,0.800000,1.000000
3,FG,2026-06-29,True,DOWN,3.0,False,False,10.0,20.0,CANDIDATE,77,6.129870,4.0,0.129870,0.311688,0.454545,0.597403,0.818182,0.961039
4,LH,2026-06-29,True,UP,7.0,False,False,40.0,20.0,CONFIRMED,14,9.571429,10.0,0.000000,0.071429,0.142857,0.214286,0.571429,1.000000
5,M,2026-06-29,False,UP,NaN,False,False,20.0,10.0,DIAGNOSTIC_ONLY,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,MA,2026-06-29,True,DOWN,5.0,False,False,40.0,20.0,CONFIRMED,32,15.937500,11.0,0.093750,0.156250,0.218750,0.375000,0.468750,0.687500
7,SA,2026-06-29,True,DOWN,13.0,False,False,20.0,10.0,CANDIDATE,17,14.941176,12.0,0.058824,0.058824,0.058824,0.235294,0.470588,0.823529
8,SP,2026-06-29,True,DOWN,3.0,True,False,10.0,20.0,CANDIDATE,58,6.086207,4.0,0.155172,0.258621,0.362069,0.568966,0.827586,1.000000
9,SR,2026-06-29,False,DOWN,NaN,False,False,60.0,20.0,CANDIDATE,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
# 输出当前截面与全历史趋势结果

REQUIRED_FINAL_OBJECTS = [
    "panel_state_duration",
    "trend_best",
    "trend_duration_summary",
    "trend_survival_table",
    "current_trend_life_report",
]
missing = [x for x in REQUIRED_FINAL_OBJECTS if x not in globals()]
if missing:
    raise RuntimeError(f"Missing required objects. Run previous cells first: {missing}")

if "panel_state" in globals():
    _state_set = set(panel_state["commodity"].unique())
    _duration_set = set(panel_state_duration["commodity"].unique())
    if _state_set != _duration_set:
        raise RuntimeError(
            f"Commodity mismatch: panel_state has {sorted(_state_set)}, "
            f"panel_state_duration has {sorted(_duration_set)}. "
            "Rerun the episode/duration/survival cell."
        )


def _select(df: pd.DataFrame, cols: list) -> pd.DataFrame:
    return df[[c for c in cols if c in df.columns]].copy()


def _p_end_key(col: str) -> int:
    return int(col.replace("p_end_within_", "").replace("d", ""))


def _reliability(n) -> str:
    if pd.isna(n) or n < 5:
        return "INSUFFICIENT"
    if n < 20:
        return "LOW"
    if n < 50:
        return "MEDIUM"
    return "HIGH"


def _trend_status(row: pd.Series) -> str:
    if bool(row.get("trend_end_confirmed", False)):
        return "TREND_CONFIRMED_ENDED"
    if not bool(row.get("trend_established_now", False)):
        return "NO_ESTABLISHED_TREND"
    side = row.get("trend_side_label", "NONE")
    return f"{side}_TREND_WITH_END_RISK" if bool(row.get("trend_end_risk", False)) else f"{side}_TREND_ESTABLISHED"


latest = (
    panel_state_duration
    .sort_values(["commodity", "date"])
    .groupby("commodity", sort=False)
    .tail(1)
    .copy()
)

p_cols = sorted(
    [c for c in current_trend_life_report.columns if c.startswith("p_end_within_")],
    key=_p_end_key,
)

life = _select(current_trend_life_report, [
    "commodity", "trend_established_now", "trend_side_label", "trend_age",
    "trend_end_risk", "trend_end_confirmed", "n_comparable_completed",
    "mean_remaining_days", "median_remaining_days",
] + p_cols)

best = _select(trend_best, [
    "commodity", "label", "h", "K", "n", "mean_y", "t_nw", "hit_rate",
    "boot_p_mean_le_0", "mean_raw", "candidate_model", "confirmed_model", "model_quality",
]).rename(columns={
    "label": "best_model_label",
    "h": "best_h",
    "K": "best_K",
    "n": "best_n",
    "mean_y": "best_mean_y",
    "t_nw": "best_t_nw",
    "hit_rate": "best_hit_rate",
    "boot_p_mean_le_0": "best_boot_p_mean_le_0",
    "mean_raw": "best_mean_raw",
    "candidate_model": "best_candidate_model",
    "confirmed_model": "best_confirmed_model",
    "model_quality": "best_model_quality",
})

duration = trend_duration_summary.copy().rename(columns={
    "n_episodes": "hist_n_episodes",
    "n_ended": "hist_n_ended",
    "n_censored": "hist_n_censored",
    "overextended_rate": "hist_overextended_rate",
    "mean_active_duration": "hist_mean_active_duration",
    "median_active_duration": "hist_median_active_duration",
    "q25_active_duration": "hist_q25_active_duration",
    "q75_active_duration": "hist_q75_active_duration",
    "mean_duration_to_end_signal": "hist_mean_duration_to_end_signal",
    "median_duration_to_end_signal": "hist_median_duration_to_end_signal",
})
duration["hist_duration_reliability"] = duration["hist_n_ended"].apply(_reliability)

# 当前Q1：趋势存在/确立
q1_existence = _select(latest, [
    "commodity", "commodity_name", "main_symbol", "date", "close",
    "h_trend_star", "K_trend_eval", "trend_level_label", "trend_model_scope",
    "TrendMainT", "TrendMainFit", "TrendMainSide",
    "TrendMainExists", "TrendEstablished",
    "TrendEpisodeID", "TrendEpisodeSideLabel", "TrendAge",
    "TrendEpisodeStartDate", "TrendEpisodeRawStartDate",
]).merge(best, on="commodity", how="left")

# 当前Q2：趋势剩余周期
q2_duration = _select(latest, [
    "commodity", "commodity_name", "main_symbol", "date", "close",
    "TrendEpisodeID", "TrendEpisodeSideLabel", "TrendAge",
    "TrendEpisodeStartDate", "TrendEpisodeRawStartDate", "TrendEpisodeCensored",
]).merge(life, on="commodity", how="left")

q2_duration["trend_side_for_duration"] = q2_duration["trend_side_label"].where(
    q2_duration["trend_established_now"].fillna(False)
)
_duration_for_merge = duration.rename(columns={"trend_side_label": "trend_side_for_duration"})
q2_duration = q2_duration.merge(
    _duration_for_merge,
    on=["commodity", "trend_side_for_duration"],
    how="left",
).drop(columns=["trend_side_for_duration"])

q2_duration["life_estimate_reliability"] = q2_duration["n_comparable_completed"].apply(_reliability)

# 当前Q3：趋势结束/风险
q3_end = _select(latest, [
    "commodity", "commodity_name", "main_symbol", "date", "close",
    "TrendMainT", "TrendMainFit", "TrendMainExists", "TrendEstablished",
    "TrendOverextendedThr", "TrendOverextended", "TrendEndRisk",
    "TrendEnd_sig", "TrendEnd_flip", "TrendEnd_ATR",
    "TrendEndConfirmed", "TrendEnd",
    "TrendEpisodeID", "TrendEpisodeEndDate", "TrendEpisodeEndReason",
]).merge(_select(life, ["commodity", "trend_end_risk", "trend_end_confirmed"] + p_cols), on="commodity", how="left")

# 当前总表
core = (
    q1_existence
    .merge(_select(q2_duration, [
        "commodity", "trend_established_now", "trend_side_label", "trend_age",
        "trend_end_risk", "trend_end_confirmed", "n_comparable_completed",
        "mean_remaining_days", "median_remaining_days", "life_estimate_reliability",
    ] + p_cols), on="commodity", how="left")
    .merge(_select(q3_end, [
        "commodity", "TrendOverextended", "TrendEndRisk", "TrendEndConfirmed", "TrendEnd",
    ]), on="commodity", how="left", suffixes=("", "_end"))
)

core["trend_status"] = core.apply(_trend_status, axis=1)
front = ["commodity", "commodity_name", "main_symbol", "date", "close", "trend_status", "life_estimate_reliability"]
core = core[[c for c in front if c in core.columns] + [c for c in core.columns if c not in front]]

# 全历史Q1：趋势频率与模型有效性
_state = panel_state_duration.copy()
_state["trend_exists_flag"] = _state["TrendMainExists"].fillna(0).astype(bool)
_state["trend_established_flag"] = _state["TrendEstablished"].fillna(0).astype(bool)
_state["trend_up_flag"] = _state["trend_established_flag"] & (_state["TrendMainSide"] > 0)
_state["trend_down_flag"] = _state["trend_established_flag"] & (_state["TrendMainSide"] < 0)

q1_history = (
    _select(_state, ["commodity", "commodity_name", "main_symbol"])
    .drop_duplicates("commodity")
    .merge(_state.groupby("commodity").agg(
        first_date=("date", "min"),
        last_date=("date", "max"),
        total_obs=("date", "size"),
        trend_exists_days=("trend_exists_flag", "sum"),
        trend_established_days=("trend_established_flag", "sum"),
        up_established_days=("trend_up_flag", "sum"),
        down_established_days=("trend_down_flag", "sum"),
    ).reset_index(), on="commodity", how="left")
    .merge(best, on="commodity", how="left")
)

q1_history["trend_exists_rate"] = q1_history["trend_exists_days"] / q1_history["total_obs"]
q1_history["trend_established_rate"] = q1_history["trend_established_days"] / q1_history["total_obs"]
q1_history["up_share_when_established"] = q1_history["up_established_days"] / q1_history["trend_established_days"].replace(0, pd.NA)
q1_history["down_share_when_established"] = q1_history["down_established_days"] / q1_history["trend_established_days"].replace(0, pd.NA)

# 全历史Q2：趋势持续时间分布
q2_history = duration.copy()

# 全历史Q3：结束风险与生存概率
q3_history = trend_survival_table.copy().rename(columns={"TrendEpisodeSideLabel": "trend_side_label"})
if {"commodity", "trend_side_label"}.issubset(q3_history.columns):
    q3_history = q3_history.merge(
        _select(duration, [
            "commodity", "trend_side_label", "hist_n_episodes", "hist_n_ended",
            "hist_overextended_rate", "hist_duration_reliability",
        ]),
        on=["commodity", "trend_side_label"],
        how="left",
    )
if "n_state_obs" in q3_history.columns:
    q3_history["survival_sample_reliability"] = q3_history["n_state_obs"].apply(_reliability)

outputs = {
    "trend_core_current_summary.csv": core,
    "trend_q1_existence_current.csv": q1_existence,
    "trend_q2_duration_current.csv": q2_duration,
    "trend_q3_end_current.csv": q3_end,
    "trend_hist_q1_existence_model.csv": q1_history,
    "trend_hist_q2_duration_distribution.csv": q2_history,
    "trend_hist_q3_end_risk_survival.csv": q3_history,
}

for filename, df_out in outputs.items():
    df_out.to_csv(RESULT_DIR / filename, index=False)

manifest = pd.DataFrame([
    {"file": "trend_core_current_summary.csv", "scope": "CURRENT", "question": "ALL", "description": "最终当前主表：趋势存在、持续周期、结束风险、最佳趋势模型汇总"},
    {"file": "trend_q1_existence_current.csv", "scope": "CURRENT", "question": "Q1", "description": "当前是否存在/确立趋势：方向、强度、拟合度、趋势年龄"},
    {"file": "trend_q2_duration_current.csv", "scope": "CURRENT", "question": "Q2", "description": "当前趋势还能持续多久：剩余寿命、未来N日结束概率、历史同方向duration摘要"},
    {"file": "trend_q3_end_current.csv", "scope": "CURRENT", "question": "Q3", "description": "当前趋势是否结束/是否有结束风险：确认结束、过度延伸、结束触发项"},
    {"file": "trend_hist_q1_existence_model.csv", "scope": "FULL_HISTORY", "question": "Q1", "description": "全历史趋势存在频率、方向分布、最佳趋势模型有效性"},
    {"file": "trend_hist_q2_duration_distribution.csv", "scope": "FULL_HISTORY", "question": "Q2", "description": "全历史趋势episode持续时间分布：均值、中位数、分位数、样本数"},
    {"file": "trend_hist_q3_end_risk_survival.csv", "scope": "FULL_HISTORY", "question": "Q3", "description": "全历史趋势结束风险：按方向、risk状态、horizon统计未来结束概率"},
])
manifest.to_csv(RESULT_DIR / "trend_output_manifest.csv", index=False)

display(core)
display(manifest)
print("Saved current + full-history trend CSVs to:", RESULT_DIR.resolve())

,commodity,commodity_name,main_symbol,date,close,trend_status,life_estimate_reliability,h_trend_star,K_trend_eval,trend_level_label,trend_model_scope,TrendMainT,TrendMainFit,TrendMainSide,TrendMainExists,TrendEstablished,TrendEpisodeID,TrendEpisodeSideLabel,TrendAge,TrendEpisodeStartDate,TrendEpisodeRawStartDate,best_model_label,best_h,best_K,best_n,best_mean_y,best_t_nw,best_hit_rate,best_boot_p_mean_le_0,best_mean_raw,best_candidate_model,best_confirmed_model,best_model_quality,trend_established_now,trend_side_label,trend_age,trend_end_risk,trend_end_confirmed,n_comparable_completed,mean_remaining_days,median_remaining_days,p_end_within_1d,p_end_within_2d,p_end_within_3d,p_end_within_5d,p_end_within_10d,p_end_within_20d,TrendOverextended,TrendEndRisk,TrendEndConfirmed,TrendEnd
0,AG,沪银,KQ.m@SHFE.ag,2026-06-29,14272.0,DOWN_TREND_ESTABLISHED,LOW,40.0,20.0,MEDIUM,CONFIRMED,-10.842801,0.755731,-1.0,1.0,1.0,AG_TREND_0344,DOWN,13.0,2026-06-10 00:00:00,2026-06-09 00:00:00,AG_Trend_h40_K20,40,20,1697,0.243651,3.107824,0.506777,0.0000,0.012075,True,True,CONFIRMED,True,DOWN,13.0,False,False,13,15.769231,15.0,0.230769,0.230769,0.230769,0.307692,0.461538,0.692308,0.0,0.0,0.0,0.0
1,CF,棉花,KQ.m@CZCE.CF,2026-06-29,16070.0,DOWN_TREND_ESTABLISHED,MEDIUM,40.0,10.0,MEDIUM,CANDIDATE,-7.037349,0.565835,-1.0,1.0,1.0,CF_TREND_0247,DOWN,9.0,2026-06-16 00:00:00,2026-06-15 00:00:00,CF_Trend_h40_K10,40,10,1766,0.139312,1.790203,0.520385,0.0125,0.002698,True,False,CANDIDATE,True,DOWN,9.0,False,False,23,16.000000,9.0,0.217391,0.217391,0.217391,0.304348,0.521739,0.739130,0.0,0.0,0.0,0.0
2,EB,苯乙烯,KQ.m@DCE.eb,2026-06-29,7249.0,DOWN_TREND_WITH_END_RISK,LOW,10.0,2.0,SHORT,CONFIRMED,-10.871930,0.936608,-1.0,1.0,1.0,EB_TREND_0133,DOWN,10.0,2026-06-15 00:00:00,2026-06-12 00:00:00,EB_Trend_h10_K2,10,2,1015,0.108912,2.079637,0.525123,0.0150,0.002190,True,True,CONFIRMED,True,DOWN,10.0,True,False,15,5.600000,3.0,0.133333,0.333333,0.533333,0.666667,0.800000,1.000000,1.0,1.0,0.0,0.0
3,FG,玻璃,KQ.m@CZCE.FG,2026-06-29,978.0,DOWN_TREND_ESTABLISHED,HIGH,10.0,20.0,SHORT,CANDIDATE,-3.904553,0.655848,-1.0,1.0,1.0,FG_TREND_0204,DOWN,3.0,2026-06-25 00:00:00,2026-06-24 00:00:00,FG_Trend_h10_K20,10,20,1550,0.115682,1.719143,0.519355,0.0175,0.003987,True,False,CANDIDATE,True,DOWN,3.0,False,False,77,6.129870,4.0,0.129870,0.311688,0.454545,0.597403,0.818182,0.961039,0.0,0.0,0.0,0.0
4,LH,生猪,KQ.m@DCE.lh,2026-06-29,12415.0,UP_TREND_ESTABLISHED,LOW,40.0,20.0,MEDIUM,CONFIRMED,5.841113,0.473090,1.0,1.0,1.0,LH_TREND_0186,UP,7.0,2026-06-18 00:00:00,2026-06-17 00:00:00,LH_Trend_h40_K20,40,20,940,0.304104,2.128568,0.550000,0.0100,0.017679,True,True,CONFIRMED,True,UP,7.0,False,False,14,9.571429,10.0,0.000000,0.071429,0.142857,0.214286,0.571429,1.000000,0.0,0.0,0.0,0.0
5,M,豆粕,KQ.m@DCE.m,2026-06-29,2967.0,NO_ESTABLISHED_TREND,INSUFFICIENT,20.0,10.0,MEDIUM,DIAGNOSTIC_ONLY,0.390346,0.008394,1.0,0.0,0.0,<NA>,<NA>,NaN,<NA>,<NA>,M_Trend_h20_K10,20,10,1814,0.054202,0.867564,0.493385,0.1800,0.001896,False,False,DIAGNOSTIC_ONLY,False,UP,NaN,False,False,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0
6,MA,甲醇,KQ.m@CZCE.MA,2026-06-29,2460.0,DOWN_TREND_ESTABLISHED,MEDIUM,40.0,20.0,MEDIUM,CONFIRMED,-6.108172,0.495417,-1.0,1.0,1.0,MA_TREND_0247,DOWN,5.0,2026-06-23 00:00:00,2026-06-22 00:00:00,MA_Trend_h40_K20,40,20,1725,0.179631,2.517070,0.513043,0.0000,0.006951,True,True,CONFIRMED,True,DOWN,5.0,False,False,32,15.937500,11.0,0.093750,0.156250,0.218750,0.375000,0.468750,0.687500,0.0,0.0,0.0,0.0
7,SA,纯碱,KQ.m@CZCE.SA,2026-06-29,1103.0,DOWN_TREND_ESTABLISHED,LOW,20.0,10.0,MEDIUM,CANDIDATE,-10.932368,0.869107,-1.0,1.0,1.0,SA_TREND_0084,DOWN,13.0,2026-06-10 00:00:00,2026-06-09 00:00:00,SA_Trend_h20_K10,20,10,1120,0.131157,1.367588,0.518750,0.0525,0.004917,True,False,CANDIDATE,True,DOWN,13.0,False,False,17,14.941176,12.0,0.058824,0.058824,0.058824,0.235294,0.470588,0.823529,0.0,0.0,0.0,0.0
8,SP,纸浆,KQ.m@SHFE.sp,2026-06-29,4690.0,DOWN_TREND_WITH_END_RISK,HIGH,10.0,20.0,SHORT,CANDIDATE,-7.208716,0.866590,-1.0,1.0,1.0,S

,file,scope,question,description
0,trend_core_current_summary.csv,CURRENT,ALL,最终当前主表：趋势存在、持续周期、结束风险、最佳趋势模型汇总
1,trend_q1_existence_current.csv,CURRENT,Q1,当前是否存在/确立趋势：方向、强度、拟合度、趋势年龄
2,trend_q2_duration_current.csv,CURRENT,Q2,当前趋势还能持续多久：剩余寿命、未来N日结束概率、历史同方向duration摘要
3,trend_q3_end_current.csv,CURRENT,Q3,当前趋势是否结束/是否有结束风险：确认结束、过度延伸、结束触发项
4,trend_hist_q1_existence_model.csv,FULL_HISTORY,Q1,全历史趋势存在频率、方向分布、最佳趋势模型有效性
5,trend_hist_q2_duration_distribution.csv,FULL_HISTORY,Q2,全历史趋势episode持续时间分布：均值、中位数、分位数、样本数
6,trend_hist_q3_end_risk_survival.csv,FULL_HISTORY,Q3,全历史趋势结束风险：按方向、risk状态、horizon统计未来结束概率


Saved current + full-history trend CSVs to: /home/zilinm2/main_continuous_daily_trend_momentum_reversal_research/results
